# T59-微信采集

本Notebook实现微信消息采集功能，包括：
- 微信群聊天记录抓取
- 微信公众号文章抓取
- 同花顺理财数据获取
- 聊天记录批处理

## 功能说明

1. **微信群消息采集**：使用wxauto库获取微信群聊天记录
2. **公众号文章抓取**：自动抓取关注的公众号最新文章
3. **理财数据更新**：从同花顺iFinD获取理财规模数据
4. **聊天记录处理**：批量处理聊天记录并保存为PDF

## 1. 环境配置与依赖导入

导入所需的库并加载配置文件。

In [ ]:
# 环境配置与依赖导入
import os
import sys
import time
import json
import re
import requests
from datetime import datetime, timedelta
from pathlib import Path

# 数据处理
import pandas as pd
import numpy as np

# 数据库
import sqlalchemy
from sqlalchemy import text
import pymysql

# 网页解析
from bs4 import BeautifulSoup

# PDF生成
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

# 微信自动化
try:
    from wxauto import WeChat
    import pyautogui
    import pyperclip
    import pygetwindow as gw
    WECHAT_AVAILABLE = True
except ImportError:
    WECHAT_AVAILABLE = False
    print("警告: wxauto未安装，微信采集功能不可用")

# 定时任务
import schedule

# 导入配置
from config import (
    get_db_connection_string,
    WECHAT_GROUPS_TO_MONITOR,
    WECHAT_CONTACTS_TO_MONITOR,
    WECHAT_PUBLIC_ACCOUNTS_TO_MONITOR,
    OUTPUT_DIR
)

print("依赖导入完成")

## 2. 数据库连接配置

配置数据库连接，用于存储采集的数据。

In [ ]:
# 数据库连接配置
def create_db_engine():
    """创建数据库连接引擎"""
    connection_string = get_db_connection_string()
    engine = sqlalchemy.create_engine(
        connection_string,
        poolclass=sqlalchemy.pool.NullPool,
        pool_recycle=3600
    )
    return engine

# 创建数据库连接
db_engine = create_db_engine()
print("数据库连接创建成功")

# 测试连接
try:
    with db_engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        print("数据库连接测试成功")
except Exception as e:
    print(f"数据库连接测试失败: {e}")

## 3. 核心功能函数

定义微信采集的核心功能函数。

In [ ]:
# 核心功能函数

class WeChatCollector:
    """微信消息采集器"""
    
    def __init__(self):
        """初始化微信采集器"""
        if not WECHAT_AVAILABLE:
            raise ImportError("wxauto未安装，无法初始化微信采集器")
        self.wx = WeChat()
        self.font_registered = False
        
    def register_chinese_font(self, font_path=None):
        """注册中文字体用于PDF生成"""
        if font_path is None:
            font_path = r'C:\Users\Administrator\AppData\Local\Microsoft\Windows\Fonts\Arial Unicode MS.ttf'
        try:
            pdfmetrics.registerFont(TTFont('ArialUnicodeMS', font_path))
            self.font_registered = True
            print(f"字体注册成功: {font_path}")
        except Exception as e:
            print(f"字体注册失败: {e}")
            
    def maximize_wechat_window(self):
        """最大化微信窗口"""
        wechat_windows = [win for win in gw.getWindowsWithTitle('微信') if win.title == '微信']
        if wechat_windows:
            wechat_windows[0].maximize()
        else:
            print("微信窗口未找到")
            
    def fetch_chat_history(self, name, is_group=True):
        """获取聊天记录
        
        Args:
            name: 群名或联系人名称
            is_group: 是否为群聊
            
        Returns:
            str: 聊天记录文本
        """
        try:
            self.wx.ChatWith(name)
            self.wx.rollToTop()
            msgs = self.wx.GetAllMessage(savepic=True)
            msg_strings = ['%s : %s\n' % (msg[0], msg[1]) for msg in msgs]
            all_msgs_str = ''.join(msg_strings)
            return all_msgs_str
        except Exception as e:
            print(f"获取聊天记录失败 [{name}]: {e}")
            return ""
            
    def save_to_pdf(self, content, filename):
        """保存内容到PDF
        
        Args:
            content: 要保存的文本内容
            filename: 输出文件名
        """
        try:
            c = canvas.Canvas(filename, pagesize=letter)
            if self.font_registered:
                c.setFont('ArialUnicodeMS', 12)
            else:
                c.setFont('Helvetica', 12)
                
            lines = content.split('\n')
            y_pos = 750
            
            for line in lines:
                # 移除图片和链接，只保留纯文本
                processed_line = ''.join(ch for ch in line if ch.isalnum() or ch.isspace() or '\u4e00' <= ch <= '\u9fff')
                c.drawString(10, y_pos, processed_line)
                y_pos -= 15
                if y_pos < 50:
                    c.showPage()
                    y_pos = 750
                    
            c.save()
            print(f"PDF保存成功: {filename}")
        except Exception as e:
            print(f"PDF保存失败: {e}")
            
    def collect_and_save_chats(self):
        """采集并保存所有监控的聊天记录"""
        now = datetime.now()
        date_str = now.strftime("%Y%m%d")
        
        # 创建输出目录
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        # 采集群聊记录
        for group in WECHAT_GROUPS_TO_MONITOR:
            chat_content = self.fetch_chat_history(group, is_group=True)
            if chat_content:
                filename = os.path.join(OUTPUT_DIR, f"{group}_{date_str}.pdf")
                self.save_to_pdf(chat_content, filename)
                
        # 采集联系人聊天记录
        for contact in WECHAT_CONTACTS_TO_MONITOR:
            chat_content = self.fetch_chat_history(contact, is_group=False)
            if chat_content:
                filename = os.path.join(OUTPUT_DIR, f"{contact}_{date_str}.pdf")
                self.save_to_pdf(chat_content, filename)

print("WeChatCollector类定义完成")

## 4. 理财数据获取

从同花顺iFinD获取理财规模数据。

In [ ]:
# 理财数据获取

def fetch_wealth_management_data():
    """从同花顺iFinD获取理财期限跟踪数据
    
    Returns:
        pd.DataFrame: 理财期限数据
    """
    try:
        from iFinDPy import THS_iFinDLogin, THS_DR
        
        # 登录iFinD
        login_result = THS_iFinDLogin(
            os.environ.get('THS_USERNAME', 'nylc082'),
            os.environ.get('THS_PASSWORD', '491448')
        )
        
        if login_result != 0:
            print("iFinD登录失败")
            return None
            
        # 获取日期
        current_date = datetime.now() - timedelta(days=1)
        current_date_str = current_date.strftime('%Y%m%d')
        dt_str = current_date.strftime('%Y-%m-%d')
        
        # 获取数据
        result = THS_DR(
            'p02186',
            f'type=收益起始日期;sdate={current_date_str};edate={current_date_str}',
            'p02186_f002:Y,p02186_f003:Y,p02186_f004:Y,p02186_f005:Y,p02186_f006:Y,'
            'p02186_f007:Y,p02186_f008:Y,p02186_f009:Y,p02186_f010:Y,p02186_f011:Y,'
            'p02186_f012:Y,p02186_f013:Y,p02186_f014:Y,p02186_f015:Y,p02186_f016:Y,'
            'p02186_f017:Y,p02186_f018:Y,p02186_f019:Y,p02186_f020:Y,p02186_f021:Y,'
            'p02186_f022:Y,p02186_f023:Y,p02186_f024:Y,p02186_f025:Y,p02186_f001:Y',
            'format:dataframe'
        )
        
        df = result.data
        if df is None:
            print(f'{dt_str}未取到数据')
            return None
            
        # 数据处理
        df = df[df['p02186_f002'] == '合计']
        df['p02186_f002'].iloc[0] = dt_str
        df = df[['p02186_f002', 'p02186_f003', 'p02186_f005', 'p02186_f006',
                 'p02186_f008', 'p02186_f009', 'p02186_f011', 'p02186_f012',
                 'p02186_f014', 'p02186_f015', 'p02186_f017', 'p02186_f018',
                 'p02186_f020', 'p02186_f021', 'p02186_f023', 'p02186_f024']]
        df.columns = ['dt', '1个月以内数量', '1个月以内占比', '1-3月数量', '1-3月占比',
                      '3-6月数量', '3-6月占比', '6-12月数量', '6-12月占比',
                      '12-24月数量', '12-24月占比', '24月以上数量', '24月以上占比',
                      '未公布数量', '未公布占比', '总数']
        
        return df
        
    except ImportError:
        print("警告: iFinDPy未安装，无法获取理财数据")
        return None
    except Exception as e:
        print(f"获取理财数据失败: {e}")
        return None

def save_wealth_data_to_db(df):
    """保存理财数据到数据库
    
    Args:
        df: 理财数据DataFrame
    """
    if df is None or df.empty:
        return
        
    try:
        with db_engine.begin() as conn:
            df.to_sql('理财期限跟踪', con=conn, if_exists='append', index=False)
        print(f"理财数据保存成功: {df['dt'].iloc[0]}")
    except Exception as e:
        print(f"保存理财数据失败: {e}")

print("理财数据获取函数定义完成")

## 5. 理财规模数据抓取

从证券之星网站抓取理财规模数据。

In [ ]:
# 理财规模数据抓取

def fetch_wealth_scale_from_web():
    """从证券之星网站抓取理财规模数据
    
    Returns:
        tuple: (规模数值, 日期)
    """
    url = "https://news.stockstar.com/author_list/张菁.shtml"
    
    try:
        response = requests.get(url, timeout=30)
        if response.status_code != 200:
            print(f"请求失败，状态码: {response.status_code}")
            return None, None
            
        response.encoding = response.apparent_encoding
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 查找张菁的文章div
        zhangjing_div = soup.find('div', class_='name', title='张菁的文章')
        if not zhangjing_div:
            print("未找到'张菁的文章'的 div")
            return None, None
            
        # 获取第一个文章链接
        first_a_tag = zhangjing_div.find_next('a')
        if not first_a_tag or 'href' not in first_a_tag.attrs:
            print("未找到链接")
            return None, None
            
        link = first_a_tag['href']
        response = requests.get(link, timeout=30)
        if response.status_code != 200:
            print(f"文章页面请求失败，状态码: {response.status_code}")
            return None, None
            
        response.encoding = response.apparent_encoding
        soup = BeautifulSoup(response.text, 'html.parser')
        text = soup.prettify()
        
        return extract_scale_and_date(text)
        
    except Exception as e:
        print(f"抓取理财规模数据失败: {e}")
        return None, None

def extract_scale_and_date(text):
    """从文本中提取规模和日期
    
    Args:
        text: 网页文本
        
    Returns:
        tuple: (规模数值, 日期)
    """
    # 查找包含关键词的子串
    pattern = re.compile(r'(截止.*?理财.*?规模.*?万亿)')
    match = pattern.search(text)
    
    if not match:
        print("未找到包含关键词的子串")
        return None, None
        
    extracted_text = text[match.start():match.start() + 50]
    print(f"提取的文本: {extracted_text}")
    
    # 提取规模数值
    number_pattern = re.compile(r'规模(.*?)万亿')
    number_match = number_pattern.search(extracted_text)
    number = None
    if number_match:
        potential_numbers = re.findall(r'[\d,\.]+', number_match.group(1))
        if potential_numbers:
            number = potential_numbers[0]
            print(f"提取的数字: {number}")
            
    # 提取日期
    date_pattern = re.compile(r'(\d{4}年\d{1,2}月\d{1,2}日)')
    date_match = date_pattern.search(extracted_text)
    formatted_date = None
    if date_match:
        date_str = date_match.group(1)
        try:
            date_obj = datetime.strptime(date_str, "%Y年%m月%d日")
            formatted_date = date_obj.strftime("%Y-%m-%d")
            print(f"提取的日期: {formatted_date}")
        except ValueError:
            print("日期格式错误")
            
    return number, formatted_date

def save_scale_to_db(number, date_str):
    """保存理财规模数据到数据库"""
    if not number or not date_str:
        return
        
    try:
        data = {'dt': [date_str], '理财规模': [number]}
        df = pd.DataFrame(data)
        with db_engine.begin() as conn:
            df.to_sql('理财规模', con=conn, if_exists='append', index=False)
        print(f"理财规模保存成功: {date_str} - {number}")
    except Exception as e:
        print(f"保存理财规模失败: {e}")

print("理财规模抓取函数定义完成")

## 6. 聊天记录批处理

批量处理聊天记录并保存。

In [ ]:
# 聊天记录批处理

def batch_process_chat_records(input_dir, output_dir):
    """批量处理聊天记录
    
    Args:
        input_dir: 输入目录
        output_dir: 输出目录
    """
    os.makedirs(output_dir, exist_ok=True)
    
    processed_count = 0
    for filename in os.listdir(input_dir):
        if filename.endswith('.txt') or filename.endswith('.pdf'):
            input_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename)
            
            try:
                # 读取并处理文件
                with open(input_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                    
                # 这里可以添加内容处理逻辑
                processed_content = content
                
                # 保存处理后的文件
                with open(output_path, 'w', encoding='utf-8') as f:
                    f.write(processed_content)
                    
                processed_count += 1
                print(f"处理完成: {filename}")
                
            except Exception as e:
                print(f"处理文件失败 [{filename}]: {e}")
                
    print(f"\n批处理完成，共处理 {processed_count} 个文件")
    return processed_count

print("聊天记录批处理函数定义完成")

## 7. 主程序执行

执行完整的数据采集流程。

In [ ]:
# 主程序执行

def main():
    """主执行函数"""
    print("=" * 50)
    print("T59-微信采集 任务开始")
    print(f"执行时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 50)
    
    # 1. 微信消息采集
    print("\n[1/4] 微信消息采集...")
    if WECHAT_AVAILABLE:
        try:
            collector = WeChatCollector()
            collector.register_chinese_font()
            collector.collect_and_save_chats()
            print("微信消息采集完成")
        except Exception as e:
            print(f"微信消息采集失败: {e}")
    else:
        print("跳过微信消息采集（wxauto未安装）")
        
    # 2. 理财数据获取
    print("\n[2/4] 理财数据获取...")
    wealth_df = fetch_wealth_management_data()
    if wealth_df is not None:
        save_wealth_data_to_db(wealth_df)
        
    # 3. 理财规模抓取
    print("\n[3/4] 理财规模抓取...")
    scale_number, scale_date = fetch_wealth_scale_from_web()
    if scale_number and scale_date:
        save_scale_to_db(scale_number, scale_date)
        
    # 4. 聊天记录批处理
    print("\n[4/4] 聊天记录批处理...")
    # batch_process_chat_records(INPUT_DIR, OUTPUT_DIR)
    print("聊天记录批处理已跳过（需要配置输入目录）")
    
    print("\n" + "=" * 50)
    print("T59-微信采集 任务完成")
    print("=" * 50)

# 执行主程序
if __name__ == '__main__':
    main()

## 总结

本Notebook实现了以下功能：

1. **微信群消息采集**：使用wxauto库自动获取微信群聊天记录
2. **理财数据获取**：从同花顺iFinD获取理财期限跟踪数据
3. **理财规模抓取**：从证券之星网站抓取理财规模数据
4. **聊天记录批处理**：批量处理聊天记录文件

### 使用说明

1. 确保已安装所有依赖包：`pip install -r requirements.txt`
2. 配置环境变量或修改config.py中的默认值
3. 运行主程序执行完整流程